# coerce-float-arg-to-array — faded example 2: coerce_pos_args: coerce real scalars across a tuple, ndarray pass-through

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `coerce-float-arg-to-array`. The last cell reports your progress on the `Backprop: Coerce float arg to array` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Coerce float arg to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`coerce-float-arg-to-array`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "coerce-float-arg-to-array"
DD_SUBTOPIC = "Backprop: Coerce float arg to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The autograd wrapper applies the per-arg scalar rule across the whole positional-args tuple: real `int`/`float` become 0-D tensors, while `bool`, `np.ndarray`, tensors, lists, and tuples pass through. Numpy arrays in particular MUST pass through — wrapping them in `t.tensor(arr)` would copy memory and promote dtype.

## Faded exercise 2

Implement `coerce_pos_args(args)`. Return a new tuple where each element is run through the per-arg rule: `bool` passes through; real `int`/`float` becomes `t.tensor(float(a))`; everything else (including `np.ndarray`) passes through unchanged. Order and length are preserved. Complete the blanked single-arg helper body.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np

def coerce_pos_args(args):
    def _coerce_one(a):
        if isinstance(a, bool):
            return a
        if isinstance(a, (int, float)):
            return t.tensor(float(a))
        return a
    return tuple(_coerce_one(a) for a in args)

np.random.seed(0)
arr = np.random.randn(3)
out = coerce_pos_args((arr, 3.0, 5, True))
print(type(out[0]).__name__, type(out[1]).__name__, float(out[1]), float(out[2]), out[3])


def _test():
    import numpy as np
    arr = np.arange(3.0)
    out = coerce_pos_args((arr, 3.0, 5, True))
    assert len(out) == 4, "length preserved"
    assert out[0] is arr, "ndarray must pass through (identity, no copy)"
    assert isinstance(out[1], t.Tensor) and out[1].ndim == 0 and float(out[1]) == 3.0
    assert out[1].dtype == t.float32
    assert isinstance(out[2], t.Tensor) and float(out[2]) == 5.0, "int coerced to float tensor"
    assert out[3] is True and isinstance(out[3], bool), "bool unchanged"
    assert coerce_pos_args(()) == (), "empty tuple"
    lst = [1, 2, 3]
    assert coerce_pos_args((lst,))[0] is lst, "list passes through (not a scalar)"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

def coerce_pos_args(args):
    def _coerce_one(a):
        if isinstance(a, bool):
            return a
        if isinstance(a, (int, float)):
            return t.tensor(float(a))
        return a
    return tuple(_coerce_one(a) for a in args)

np.random.seed(0)
arr = np.random.randn(3)
out = coerce_pos_args((arr, 3.0, 5, True))
print(type(out[0]).__name__, type(out[1]).__name__, float(out[1]), float(out[2]), out[3])
```
</details>